# 03. NER y Análisis de Sentimiento

Extracción de entidades nombradas y análisis de polaridad emocional.


In [ ]:
import pandas as pd
import spacy
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
from pathlib import Path
from tqdm.notebook import tqdm

DATA_INTERIM = Path("data/interim")
df = pd.read_csv(DATA_INTERIM / "02_preprocesado.csv")
print(f"Cargado: {len(df):,} filas")


In [ ]:
# NER con spaCy + reglas de dominio
nlp_ner = spacy.load("en_core_web_sm")
ruler = nlp_ner.add_pipe("entity_ruler", before="ner", config={"overwrite_ents": True})

financial = ["Equifax", "Experian", "TransUnion", "CFPB", "IRS", "FTC", "PennyMac", "Navient", "Chase", "Bank of America", "Wells Fargo", "Capital One", "Discover", "Synchrony"]
patterns = [{"label": "ORG", "pattern": e} for e in financial]
patterns += [
    {"label": "LAW", "pattern": "FCRA"},
    {"label": "LAW", "pattern": "FDCPA"},
    {"label": "LAW", "pattern": [{"LOWER": "section"}, {"LIKE_NUM": True}]},
    {"label": "LAW", "pattern": [{"LOWER": "fair"}, {"LOWER": "credit"}, {"LOWER": "reporting"}, {"LOWER": "act"}]},
]
ruler.add_patterns(patterns)

def extract_entities(text):
    if not isinstance(text, str): return {}
    doc = nlp_ner(text)
    ents = {}
    for ent in doc.ents:
        if "[MASK]" in ent.text: continue
        ents.setdefault(ent.label_, []).append(ent.text)
    return {k: sorted(list(set(v))) for k, v in ents.items()}

df["entities"] = [extract_entities(t) for t in tqdm(df["Consumer complaint narrative"].astype(str))]
df["entity_count"] = df["entities"].apply(lambda x: sum(len(v) for v in x.values()) if isinstance(x, dict) else 0)
print("NER completado.")


In [ ]:
# Sentimiento
vader = SentimentIntensityAnalyzer()

def vader_analysis(text):
    s = vader.polarity_scores(text)
    c = s["compound"]
    label = "Muy Negativo" if c <= -0.5 else "Negativo" if c < -0.05 else "Neutral" if c <= 0.05 else "Positivo" if c < 0.5 else "Muy Positivo"
    return s["compound"], s["neg"], s["neu"], s["pos"], label

def textblob_analysis(text):
    b = TextBlob(text)
    p = b.sentiment.polarity
    label = "Muy Negativo" if p <= -0.3 else "Negativo" if p < -0.05 else "Neutral" if p <= 0.05 else "Positivo" if p < 0.3 else "Muy Positivo"
    return p, b.sentiment.subjectivity, label

v_res = [vader_analysis(t) for t in tqdm(df["Consumer complaint narrative"].astype(str))]
tb_res = [textblob_analysis(t) for t in tqdm(df["Consumer complaint narrative"].astype(str))]

df["vader_compound"] = [r[0] for r in v_res]
df["vader_label"] = [r[4] for r in v_res]
df["textblob_polarity"] = [r[0] for r in tb_res]
df["textblob_label"] = [r[2] for r in tb_res]

df.to_csv(DATA_INTERIM / "03_ner_sentimiento.csv", index=False)
print("Sentimiento completado.")
